<a href="https://colab.research.google.com/github/JorgeSanchezCano/curso-llama/blob/main/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [91]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [92]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'La comida tardó pero el sabor es muy bueno.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto.


In [93]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Muy sabroso, Sirvieron rápido y el lugar es muy limpio."
Sentimiento: Positivo

Reseña: "llego frio, mal sabor , pésimo servicio."
Sentimiento: Negativo

Reseña: "pesimo servicio pero el producto es Muy sabroso."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [94]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un auto sale de la ciudad A a 50 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 150 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Pasos del razonamiento**

1. **Comprender la situación inicial**  
   - Vehículo 1 (el “auto”): sale de la ciudad A a 50 km/h.  
   - 2 horas después, Vehículo 2 (el “otro tren”) sale de la misma ciudad a 150 km/h.

2. **Calcular la ventaja que tiene el primer vehículo cuando el segundo partió**  
   - Tiempo de ventaja = 2 h.  
   - Velocidad del vehículo 1 = 50 km/h.  
   - Distancia que recorre el vehículo 1 en ese tiempo:  
     \[
     d_{\text{ventaja}} = v_1 \times t = 50\,\text{km/h} \times 2\,\text{h} = 100\,\text{km}.
     \]

3. **Determinar la velocidad relativa**  
   - Cuando el segundo vehículo está en marcha, ambos viajan en la misma dirección.  
   - Velocidad del vehículo 2 = 150 km/h.  
   - Velocidad relativa (diferencia de velocidades) =  
     \[
     v_{\text{rel}} = v_2 - v_1 = 150\,\text{km/h} - 50\,\text{km/h} = 100\,\text{km/h}.
     \]

4. **Calcular el tiempo necesario para cerrar la brecha**  
   - Brecha inicial = 100 km.  
   - Tiempo para cerrar esa b

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [99]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final de las evaluaciones industriales de la compaña ABC del año 2018"
    "? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

No dispongo de esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [100]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [18]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings


In [97]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


In [45]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [101]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

Se aplica un descuento del 20%.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [102]:
# Leer API key, instalar e importar librerías
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [103]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Los productos disponibles para venta son ",
    "tacos de pibil precio 12 pesos, tacos de cuerito precio 12 pesos, tacos de barbacoa, tacos de cabeza precio 12 pesos ",
    "Los envíos a casa tienen un costo extra de 20 pesos  ",
    "Si el pedido rebasa los 240 pesos el envío es gratis ",
    "Si el pedido tarda mas de 50 minutos se da un descuento del 20%"
]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (5, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [104]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "Si el pedido se retraso que aplica?"
fragmento = buscar_fragmento(pregunta)
fragmento2 = " "
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Si el pedido tarda mas de 50 minutos se da un descuento del 20%


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [105]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
prompt_rag = f"""...


Política: {fragmento2}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_sin_rag.choices[0].message.content)

Aplica la política de compensación por retraso.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [106]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_con_rag.choices[0].message.content)

Descuento del 20%.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [107]:
# Mostrar ambas respuestas para comparar
print("Sin RAG ->", response_sin_rag.choices[0].message.content)
print("Con RAG ->", response_con_rag.choices[0].message.content)


Sin RAG -> Aplica la política de compensación por retraso.
Con RAG -> Descuento del 20%.
